# Smoke tests de OP-11 `read_flows`

Este notebook contiene smoke tests integrados de la operación pública `read_flows()`.

Objetivo:

- verificar que la lectura formal de bundles `.golondrina` funcione en escenarios simples y representativos;
- comprobar que el sidecar obligatorio se usa como fuente de verdad;
- revisar la reconstrucción mínima de `FlowDataset`;
- verificar la política post-read de `metadata["is_validated"] = False`;
- comprobar el manejo opcional de `flow_to_trips`;
- revisar el comportamiento de `keep_metadata`;
- cubrir una falla fatal por sidecar ausente.

Algunos tests usan previamente `write_flows()` para crear un artefacto formal válido que luego es leído por `read_flows()`. En esos casos, el foco del test sigue estando en la operación de lectura.

Los artefactos de persistencia se crean en una carpeta local junto al notebook:
`./tmp_smoke_read_flows`.

## Bloque 1. Setup visible de smoke tests

Qué prepara:

- una carpeta local visible para inspeccionar artefactos persistidos;
- factories pequeñas para construir `FlowDataset`;
- helpers de lectura de sidecar y códigos de issues;
- imports de `write_flows` y `read_flows`;
- datos sintéticos mínimos reutilizables en todos los smoke tests.

In [1]:
from pathlib import Path
import json
import shutil
import copy

import pandas as pd

from pylondrina.datasets import FlowDataset
from pylondrina.errors import ExportError
from pylondrina.io.flows import (
    write_flows,
    read_flows,
    WriteFlowsOptions,
    ReadFlowsOptions,
)


SMOKE_ROOT = Path("./tmp_smoke_read_flows")


def show_ok(label: str):
    print(f"OK - {label}")


def reset_smoke_root() -> Path:
    if SMOKE_ROOT.exists():
        shutil.rmtree(SMOKE_ROOT)
    SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
    return SMOKE_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = SMOKE_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


def flow_issue_codes(report) -> list[str]:
    return [issue.code for issue in report.issues]


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def make_flow_df(n_repeat: int = 1) -> pd.DataFrame:
    base = pd.DataFrame(
        {
            "flow_id": ["f_0001", "f_0002", "f_0003"],
            "origin_h3_index": [
                "8828308281fffff",
                "8828308281fffff",
                "8828308285fffff",
            ],
            "destination_h3_index": [
                "8828308287fffff",
                "8828308289fffff",
                "8828308287fffff",
            ],
            "flow_count": [10, 6, 4],
            "flow_value": [10.0, 6.0, 4.0],
            "mode": ["bus", "metro", "bus"],
            "window_start_utc": pd.to_datetime(
                [
                    "2026-01-01T08:00:00Z",
                    "2026-01-01T08:00:00Z",
                    "2026-01-01T09:00:00Z",
                ],
                utc=True,
            ),
            "window_end_utc": pd.to_datetime(
                [
                    "2026-01-01T09:00:00Z",
                    "2026-01-01T09:00:00Z",
                    "2026-01-01T10:00:00Z",
                ],
                utc=True,
            ),
        }
    )

    if n_repeat <= 1:
        return base

    parts = []
    for i in range(n_repeat):
        part = base.copy(deep=True)
        part["flow_id"] = [
            f"{fid}_r{i:04d}"
            for fid in part["flow_id"]
        ]
        parts.append(part)

    return pd.concat(parts, ignore_index=True)


def make_flow_to_trips_df(
    flow_ids: list[str] | None = None,
) -> pd.DataFrame:
    if flow_ids is None:
        flow_ids = ["f_0001", "f_0002", "f_0003"]

    rows = []
    for idx, fid in enumerate(flow_ids):
        rows.append(
            {
                "flow_id": fid,
                "movement_id": f"m_{idx * 2 + 1:04d}",
            }
        )
        rows.append(
            {
                "flow_id": fid,
                "movement_id": f"m_{idx * 2 + 2:04d}",
            }
        )

    return pd.DataFrame(rows)


def make_flowdataset(
    *,
    validated: bool = True,
    with_flow_to_trips: bool = False,
    dataset_id: str = "flow-dset-smoke-001",
) -> FlowDataset:
    flows_df = make_flow_df()

    flow_to_trips_df = (
        make_flow_to_trips_df(flows_df["flow_id"].tolist())
        if with_flow_to_trips
        else None
    )

    aggregation_spec = {
        "h3_resolution": 8,
        "group_by": ["mode"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "min_trips_per_flow": 1,
    }

    metadata = {
        "dataset_id": dataset_id,
        "is_validated": validated,
        "events": [],
        "notes": {"smoke_case": True},
    }

    provenance = {
        "derived_from": [
            {
                "source_type": "trips",
                "dataset_id": "trip-dset-origin-001",
            }
        ],
        "prior_events_summary": {"n_events": 2},
    }

    return FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips_df,
        aggregation_spec=aggregation_spec,
        source_trips={"debug_only": True},
        metadata=metadata,
        provenance=provenance,
    )


root = reset_smoke_root()
print("SMOKE_ROOT =", root.resolve())
show_ok("Bloque 1 - setup visible de smoke tests de OP-11 read_flows")

SMOKE_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_smoke_read_flows
OK - Bloque 1 - setup visible de smoke tests de OP-11 read_flows


## Bloque 2. Smoke test de `read_flows` happy path mínimo usando fallback `.golondrina`

Qué prueba:

- lectura formal exitosa a partir de un artefacto correcto;
- uso del fallback `path + ".golondrina"` cuando el path exacto no existe;
- reconstrucción de `FlowDataset`;
- ausencia de `source_trips` tras leer;
- forzado de `metadata["is_validated"] = False`;
- incorporación del evento `read_flows`;
- `summary` y `parameters` coherentes.

Se usa previamente `write_flows()` para dejar un bundle formal válido. Como no se indica `storage_format`, la escritura usa el backend por defecto vigente: **Feather**. Por ello, la lectura debe reportar `flows.feather` entre los archivos leídos. 

In [2]:
case_dir = make_case_dir("case_01_read_happy_from_sidecar")
artifact_dir = case_dir / "artifact"  # sin sufijo
true_artifact_dir = case_dir / "artifact.golondrina"

flows = make_flowdataset(
    validated=True,
    with_flow_to_trips=False,
)

write_report = write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)

loaded, read_report = read_flows(
    artifact_dir,  # intencionalmente sin ".golondrina"
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

assert write_report.ok is True
assert read_report.ok is True
assert true_artifact_dir.exists()

# Summary / parameters
assert read_report.summary["n_flows"] == len(flows.flows)
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None

assert "flows.feather" in read_report.summary["files_read"]
assert "flows.metadata.json" in read_report.summary["files_read"]

assert read_report.parameters["path"] == str(true_artifact_dir)
assert read_report.parameters["strict"] is False
assert read_report.parameters["keep_metadata"] is True
assert read_report.parameters["read_flow_to_trips"] is False

# Post-read
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_flows" in ops_loaded
assert ops_loaded[-1] == "read_flows"

pd.testing.assert_frame_equal(
    loaded.flows.reset_index(drop=True),
    flows.flows.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

display(read_report)
show_ok("Bloque 2 - read_flows happy path mínimo con fallback .golondrina")

OperationReport(ok=True, issues=[Issue(level='info', code='READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE', message="El estado metadata['is_validated'] fue forzado a False al finalizar read_flows.", field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke_read_flows\\case_01_read_happy_from_sidecar\\artifact.golondrina', 'strict': False, 'reason': 'force_unvalidated_after_read', 'recovered': True, 'recovery_action': 'force_is_validated_false'})], summary={'n_flows': 3, 'n_columns': 8, 'flow_to_trips_loaded': False, 'n_flow_to_trips': None, 'files_read': ['flows.feather', 'flows.metadata.json'], 'dataset_id': 'flow-dset-smoke-001', 'artifact_id': 'art_c5decc6e-c08e-4d70-af44-a8e113c95c97'}, parameters={'path': 'tmp_smoke_read_flows\\case_01_read_happy_from_sidecar\\artifact.golondrina', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': False})

OK - Bloque 2 - read_flows happy path mínimo con fallback .golondrina


## Bloque 3. Smoke test integrado de write/read con auxiliar `flow_to_trips` existente

Qué prueba:

- persistencia correcta del auxiliar opcional;
- reconstrucción correcta de `flow_to_trips` en `read_flows`;
- `flow_to_trips_loaded=True`;
- conteo correcto de filas auxiliares;
- igualdad esencial entre el auxiliar original y el reconstruido.

Como `write_flows()` se invoca sin `storage_format`, el backend efectivo es **Feather**, por lo que se espera `flow_to_trips.feather`.

In [3]:
case_dir = make_case_dir("case_02_with_aux_present")
artifact_dir = case_dir / "artifact"

flows = make_flowdataset(
    validated=True,
    with_flow_to_trips=True,
)

write_report = write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

loaded, read_report = read_flows(
    artifact_dir,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert (artifact_dir / "flow_to_trips.feather").exists()
assert "flow_to_trips.feather" in write_report.summary["files_written"]

assert read_report.summary["flow_to_trips_loaded"] is True
assert read_report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)

assert "flow_to_trips.feather" in read_report.summary["files_read"]

assert loaded.flow_to_trips is not None

pd.testing.assert_frame_equal(
    loaded.flow_to_trips.reset_index(drop=True),
    flows.flow_to_trips.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

display(read_report)
show_ok("Bloque 3 - write/read con flow_to_trips existente")

OperationReport(ok=True, issues=[Issue(level='info', code='READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE', message="El estado metadata['is_validated'] fue forzado a False al finalizar read_flows.", field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke_read_flows\\case_02_with_aux_present\\artifact', 'strict': False, 'reason': 'force_unvalidated_after_read', 'recovered': True, 'recovery_action': 'force_is_validated_false'})], summary={'n_flows': 3, 'n_columns': 8, 'flow_to_trips_loaded': True, 'n_flow_to_trips': 6, 'files_read': ['flows.feather', 'flow_to_trips.feather', 'flows.metadata.json'], 'dataset_id': 'flow-dset-smoke-001', 'artifact_id': 'art_16da4ebe-9eaf-40df-8675-789fd464b2b5'}, parameters={'path': 'tmp_smoke_read_flows\\case_02_with_aux_present\\artifact', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': True})

OK - Bloque 3 - write/read con flow_to_trips existente


## Bloque 4. Smoke test degradado de `read_flows` con auxiliar solicitado pero faltante

Qué prueba:

- caso recuperable bajo `strict=False`;
- el sidecar formal existe y el bundle principal sigue siendo legible;
- se solicita cargar `flow_to_trips`;
- el archivo auxiliar fue removido manualmente;
- `read_flows` no aborta;
- emite el issue `READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING`;
- devuelve `flow_to_trips=None`.

El contrato vigente fija que la ausencia del auxiliar puede degradarse bajo `strict=False`, mientras que el sidecar formal sigue siendo obligatorio. 

In [4]:
case_dir = make_case_dir("case_03_read_degraded_missing_aux")
artifact_dir = case_dir / "artifact"

flows = make_flowdataset(
    validated=True,
    with_flow_to_trips=True,
)

write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

# Dejo el artefacto formal, pero quito manualmente el auxiliar.
(artifact_dir / "flow_to_trips.feather").unlink()

loaded, report = read_flows(
    artifact_dir,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = flow_issue_codes(report)

assert report.ok is True
assert "READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING" in codes

assert loaded.flow_to_trips is None
assert report.summary["flow_to_trips_loaded"] is False
assert report.summary["n_flow_to_trips"] is None

display(report)
show_ok("Bloque 4 - read_flows degradado por auxiliar faltante con strict=False")

OperationReport(ok=True, issues=[Issue(level='warning', code='READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING', message='Se solicitó cargar flow_to_trips, pero el archivo no existe; la lectura continuará sin auxiliar bajo strict=False.', field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke_read_flows\\case_03_read_degraded_missing_aux\\artifact', 'read_flow_to_trips': True, 'files_expected': ['flow_to_trips.feather'], 'files_read': ['flows.feather', 'flows.metadata.json'], 'reason': 'missing_flow_to_trips_file', 'recovered': True, 'recovery_action': 'omit_missing_flow_to_trips'}), Issue(level='info', code='READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE', message="El estado metadata['is_validated'] fue forzado a False al finalizar read_flows.", field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke_read_flows\\case_03_read_degraded_missing_aux\\artifact', 'strict': False, 'reason': 'force_unvalidated_after_read', 'recovered': True, 'recovery_action':

OK - Bloque 4 - read_flows degradado por auxiliar faltante con strict=False


## Bloque 5. Smoke test de `read_flows` con `keep_metadata=False`

Qué prueba:

- el dataset se reconstruye normalmente;
- `dataset_id`, `artifact_id` y `provenance` se preservan;
- `metadata["is_validated"]` queda forzado a `False`;
- no se agrega el evento `read_flows`;
- la metadata persistida no se poda ni se reemplaza, solo se omite el nuevo evento.

La implementación vigente de OP-11 interpreta `keep_metadata=False` como “no append del evento `read_flows`”, no como eliminación de metadata persistida. :contentReference[oaicite:4]{index=4}


In [5]:
case_dir = make_case_dir("case_04_read_keep_metadata_false")
artifact_dir = case_dir / "artifact"

flows = make_flowdataset(
    validated=True,
    with_flow_to_trips=False,
)

write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

loaded, report = read_flows(
    artifact_dir,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=False,
        read_flow_to_trips=False,
    ),
)

assert report.ok is True

assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.provenance == flows.provenance

ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert ops_loaded[-1] == "write_flows"  # no se agregó read_flows

display(report)
show_ok("Bloque 5 - read_flows con keep_metadata=False")

OperationReport(ok=True, issues=[Issue(level='info', code='READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE', message="El estado metadata['is_validated'] fue forzado a False al finalizar read_flows.", field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke_read_flows\\case_04_read_keep_metadata_false\\artifact', 'strict': False, 'reason': 'force_unvalidated_after_read', 'recovered': True, 'recovery_action': 'force_is_validated_false'})], summary={'n_flows': 3, 'n_columns': 8, 'flow_to_trips_loaded': False, 'n_flow_to_trips': None, 'files_read': ['flows.feather', 'flows.metadata.json'], 'dataset_id': 'flow-dset-smoke-001', 'artifact_id': 'art_2616640f-3db5-453e-9208-bf5c778c8a8e'}, parameters={'path': 'tmp_smoke_read_flows\\case_04_read_keep_metadata_false\\artifact', 'strict': False, 'keep_metadata': False, 'read_flow_to_trips': False})

OK - Bloque 5 - read_flows con keep_metadata=False


## Bloque 6. Smoke test fatal de `read_flows` por layout inválido

Qué prueba:

- existe un directorio con una tabla física de flows;
- falta `flows.metadata.json`;
- la lectura formal no debe aceptar ese artefacto;
- se lanza `ExportError`.

Este smoke test verifica una decisión central de OP-11: el sidecar `flows.metadata.json` es obligatorio y su ausencia **no es recuperable**. 

In [6]:
case_dir = make_case_dir("case_05_read_fatal_missing_sidecar")
artifact_dir = case_dir / "artifact"
artifact_dir.mkdir(parents=True, exist_ok=True)

# Creo solo la tabla principal, sin sidecar formal.
# El tipo físico de la tabla no cambia la lógica del test:
# el error debe venir por sidecar ausente.
make_flow_df().to_parquet(
    artifact_dir / "flows.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

raised = None

try:
    read_flows(
        artifact_dir,
        options=ReadFlowsOptions(
            strict=False,
            keep_metadata=True,
            read_flow_to_trips=False,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)

display(raised)
show_ok("Bloque 6 - fatal de read_flows por sidecar faltante")

ExportError(message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', details={'path': 'tmp_smoke_read_flows\\case_05_read_fatal_missing_sidecar\\artifact', 'files_expected': ['flows.metadata.json'], 'reason': 'missing_flows_metadata_json', 'action': 'abort'}, issue=Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke_read_flows\\case_05_read_fatal_missing_sidecar\\artifact', 'files_expected': ['flows.metadata.json'], 'reason': 'missing_flows_metadata_json', 'action': 'abort'}), issues=(Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', field=None, source_field=None, row_count

OK - Bloque 6 - fatal de read_flows por sidecar faltante


## Bloque 7. Smoke test integrado de round-trip completo `write_flows` + `read_flows`

Qué prueba:

- persistencia y reconstrucción completa de un `FlowDataset` pequeño;
- igualdad esencial de:
  - `flows`,
  - `flow_to_trips`,
  - `aggregation_spec`,
  - `provenance`;
- conservación de `dataset_id` y `artifact_id`;
- `source_trips=None` después de leer;
- `metadata["is_validated"] = False` post-read;
- eventos `write_flows` y `read_flows` presentes y ordenados.

Este bloque usa explícitamente Parquet para mantener una ruta de round-trip distinta de los smoke tests anteriores, que ejercitan el backend Feather por defecto.

In [7]:
case_dir = make_case_dir("case_06_roundtrip_full")
artifact_dir = case_dir / "artifact"

flows_original = make_flowdataset(
    validated=True,
    with_flow_to_trips=True,
)

flows_df_original = flows_original.flows.copy(deep=True)
flow_to_trips_original = flows_original.flow_to_trips.copy(deep=True)
aggregation_spec_original = copy.deepcopy(flows_original.aggregation_spec)
provenance_original = copy.deepcopy(flows_original.provenance)

print("Datos originales en memoria")
display(flows_original.source_trips)
display(flows_original.flows)
display(flows_original.flow_to_trips)

write_report = write_flows(
    flows_original,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

loaded, read_report = read_flows(
    artifact_dir,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

# Identidad
assert loaded.metadata["dataset_id"] == flows_original.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows_original.metadata["artifact_id"]

# Regla post-read
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

# Data y auxiliar esencialmente iguales
pd.testing.assert_frame_equal(
    loaded.flows.reset_index(drop=True),
    flows_df_original.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

pd.testing.assert_frame_equal(
    loaded.flow_to_trips.reset_index(drop=True),
    flow_to_trips_original.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

# Objetos serializables equivalentes
assert loaded.aggregation_spec == aggregation_spec_original
assert loaded.provenance == provenance_original

# Eventos: write persistido + read append
ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_flows" in ops_loaded
assert ops_loaded[-1] == "read_flows"

print("ops_loaded =", ops_loaded)

display(write_report.summary)
display(read_report.summary)

print("Datos reconstruidos tras read")
display(loaded.source_trips)
display(loaded.flows)
display(loaded.flow_to_trips)

show_ok("Bloque 7 - round-trip completo write_flows/read_flows")

Datos originales en memoria


{'debug_only': True}

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,window_start_utc,window_end_utc
0,f_0001,8828308281fffff,8828308287fffff,10,10.0,bus,2026-01-01 08:00:00+00:00,2026-01-01 09:00:00+00:00
1,f_0002,8828308281fffff,8828308289fffff,6,6.0,metro,2026-01-01 08:00:00+00:00,2026-01-01 09:00:00+00:00
2,f_0003,8828308285fffff,8828308287fffff,4,4.0,bus,2026-01-01 09:00:00+00:00,2026-01-01 10:00:00+00:00


,flow_id,movement_id
0,f_0001,m_0001
1,f_0001,m_0002
2,f_0002,m_0003
3,f_0002,m_0004
4,f_0003,m_0005
5,f_0003,m_0006


ops_loaded = ['write_flows', 'read_flows']


...

Datos reconstruidos tras read


None

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,window_start_utc,window_end_utc
0,f_0001,8828308281fffff,8828308287fffff,10,10.0,bus,2026-01-01 08:00:00+00:00,2026-01-01 09:00:00+00:00
1,f_0002,8828308281fffff,8828308289fffff,6,6.0,metro,2026-01-01 08:00:00+00:00,2026-01-01 09:00:00+00:00
2,f_0003,8828308285fffff,8828308287fffff,4,4.0,bus,2026-01-01 09:00:00+00:00,2026-01-01 10:00:00+00:00


,flow_id,movement_id
0,f_0001,m_0001
1,f_0001,m_0002
2,f_0002,m_0003
3,f_0002,m_0004
4,f_0003,m_0005
5,f_0003,m_0006


OK - Bloque 7 - round-trip completo write_flows/read_flows
